# Lecture 4 — Class Exercise
## Scatter & Bubble Charts: Gapminder

> **Push to:** `week04/lecture04_exercise.ipynb`

**Rules:**
1. Colour used **sparingly** — one categorical variable, no rainbow
2. If showing all continents, either use accessible palette OR grey all + highlight one
3. `size_max` set when using bubble size
4. Log scale for GDP per capita
5. Insight title

---


In [1]:
import pandas as pd
import plotly.express as px


# Dataset: Gapminder — GDP, Life Expectancy, Population by Country
# Source: Gapminder Foundation (gapminder.org)

df = px.data.gapminder()
print(f"Loaded: {len(df)} rows")
print(df.head())

Loaded: 1704 rows
       country continent  year  lifeExp       pop   gdpPercap iso_alpha  \
0  Afghanistan      Asia  1952   28.801   8425333  779.445314       AFG   
1  Afghanistan      Asia  1957   30.332   9240934  820.853030       AFG   
2  Afghanistan      Asia  1962   31.997  10267083  853.100710       AFG   
3  Afghanistan      Asia  1967   34.020  11537966  836.197138       AFG   
4  Afghanistan      Asia  1972   36.088  13079460  739.981106       AFG   

   iso_num  
0        4  
1        4  
2        4  
3        4  
4        4  


In [2]:
# explore

print(df.info())
print("Years:", sorted(df['year'].unique()))
print("Continents:", df['continent'].unique())
print(df.describe().round(1))


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1704 entries, 0 to 1703
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   country    1704 non-null   object 
 1   continent  1704 non-null   object 
 2   year       1704 non-null   int64  
 3   lifeExp    1704 non-null   float64
 4   pop        1704 non-null   int64  
 5   gdpPercap  1704 non-null   float64
 6   iso_alpha  1704 non-null   object 
 7   iso_num    1704 non-null   int64  
dtypes: float64(2), int64(3), object(3)
memory usage: 106.6+ KB
None
Years: [np.int64(1952), np.int64(1957), np.int64(1962), np.int64(1967), np.int64(1972), np.int64(1977), np.int64(1982), np.int64(1987), np.int64(1992), np.int64(1997), np.int64(2002), np.int64(2007)]
Continents: ['Asia' 'Europe' 'Africa' 'Americas' 'Oceania']
         year  lifeExp           pop  gdpPercap  iso_num
count  1704.0   1704.0  1.704000e+03     1704.0   1704.0
mean   1979.5     59.5  2.960121e+07     7215.3    

## Task 1 — Scatter: life expectancy change over time

**What to build:** A scatter showing **GDP per capita vs life expectancy** for **two years** (2002 and 2007) to show how both moved — use **colour for year** (just 2 colours), **one continent only**.

Choose any continent except Africa (that was the example). Highlight the change direction.

> 💡 Filter: `df.loc[df['continent'] == 'YOUR_CHOICE']` then filter years


In [5]:
# Task 1
# YOUR CODE HERE

# Step 1: Filter one continent and the two years
asia = df.loc[(df['continent'] == 'Asia') & (df['year'].isin([2002, 2007]))].copy()
asia['year'] = asia['year'].astype(str)   # year as category -> 2 colours

# Step 2: Count how many countries improved (for the title)
wide = asia.pivot(index='country', columns='year', values=['gdpPercap', 'lifeExp'])
n = len(wide)
richer = (wide['gdpPercap']['2007'] > wide['gdpPercap']['2002']).sum()

# Step 3: Plot both years — sequential blues: light = before, dark = after
fig = px.scatter(
    asia,
    x='gdpPercap',
    y='lifeExp',
    color='year',
    color_discrete_map={'2002': '#9ECAE1', '2007': '#08519C'},
    hover_name='country',
    log_x=True,   # log scale for GDP
    labels={'gdpPercap': 'GDP per capita (US$, log scale)',
            'lifeExp': 'Life expectancy (years)', 'year': ''}
)
fig.update_traces(marker=dict(size=10, line=dict(width=1, color='white')))

# Step 4: Connect each country's 2002 -> 2007 dot to show the direction of change
xs, ys = [], []
for country, row in wide.iterrows():
    xs += [row[('gdpPercap', '2002')], row[('gdpPercap', '2007')], None]
    ys += [row[('lifeExp', '2002')], row[('lifeExp', '2007')], None]
fig.add_trace(go.Scatter(x=xs, y=ys, mode='lines', line=dict(color='#C6DBEF', width=1.5),
                         hoverinfo='skip', showlegend=False))
fig.data = fig.data[-1:] + fig.data[:-1]   # lines underneath the dots

# Step 5: Layout improvements
fig.update_layout(
    title=f"Between 2002 and 2007, all {n} Asian countries gained life expectancy — and {richer} of {n} got richer",
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=13, color='#333333'),
    xaxis=dict(gridcolor='#F0F0F0'),
    yaxis=dict(gridcolor='#F0F0F0'),
    legend=dict(orientation='h', y=1.06, x=0),
    height=550
)

fig.show()

## Task 2 — Bubble chart: tell a story

**What to build:** A bubble chart (full 2007 dataset, all countries) where:
- x = GDP per capita (log scale)
- y = life expectancy
- size = population
- colour = ONE continent highlighted (your choice), all others grey
- At least one annotation explaining the highlighted group's story

> This is the grey-and-highlight technique applied to a bubble chart.


In [11]:
# Task 2
# YOUR CODE HERE

# Step 1: 2007 data + highlight column
d07 = df.loc[df['year'] == 2007].copy()
d07['highlight'] = d07['continent'].apply(lambda c: 'Africa' if c == 'Africa' else 'Rest of world')   # CHANGED

# Step 2: Numbers for the story
africa = d07[d07['highlight'] == 'Africa']
under60_africa = (africa['lifeExp'] < 60).sum()
under60_rest = (d07[d07['highlight'] == 'Rest of world']['lifeExp'] < 60).sum()   # CHANGED

# Step 3: Bubble chart — Africa orange, everyone else grey
fig = px.scatter(
    d07.sort_values('highlight', ascending=False),   # grey drawn first, Africa on top
    x='gdpPercap',
    y='lifeExp',
    size='pop',
    size_max=50,   # cap bubble size
    color='highlight',
    color_discrete_map={'Africa': '#E07B39', 'Rest of world': '#D0D0D0'},   # CHANGED
    category_orders={'highlight': ['Africa', 'Rest of world']},             # NEW: Africa first in legend
    hover_name='country',
    log_x=True,    # log scale for GDP
    labels={'gdpPercap': 'GDP per capita (US$, log scale)',
            'lifeExp': 'Life expectancy (years)',
            'highlight': ''}                                                 # NEW: no legend title
)
fig.update_traces(marker=dict(opacity=0.85, line=dict(width=0.5, color='white')))

# Step 4: Annotations telling the story
fig.add_hline(y=60, line_dash='dot', line_color='#999')

fig.add_annotation(
    x=np.log10(20000), y=44,   # log axis -> position uses log10
    text=f"<b>Africa:</b> {under60_africa} of {len(africa)} countries live under 60 years<br>— only {under60_rest} countries elsewhere do",
    showarrow=False, align='left',
    font=dict(color='#E07B39', size=12)
)

bw = africa[africa['country'] == 'Botswana'].iloc[0]
fig.add_annotation(
    x=np.log10(bw['gdpPercap']), y=bw['lifeExp'],
    text='Botswana: $12.6k GDP,<br>yet only 51 years',
    showarrow=True, arrowhead=2, arrowcolor='#E07B39', ax=60, ay=-55,
    font=dict(color='#E07B39', size=11)
)

# Step 5: Layout improvements
fig.update_layout(
    title="Africa is stuck at the bottom-left: low income and the world's shortest lives",
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=13),
    xaxis=dict(gridcolor='#F0F0F0'),
    yaxis=dict(gridcolor='#F0F0F0'),
    showlegend=True,                                                    # CHANGED
    legend=dict(orientation='h', x=0, y=1.06, itemsizing='constant'),   # NEW
    height=600
)

fig.show()